In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from PIL import Image
import shutil

TRAIN_DIR_RAW = "/content/drive/MyDrive/CMPE401/YOLO/VisDrone2019-DET-train"
VAL_DIR_RAW = "/content/drive/MyDrive/CMPE401/YOLO/VisDrone2019-DET-val"
TEST_DIR_RAW = "/content/drive/MyDrive/CMPE401/YOLO/VisDrone2019-DET-test-dev"

DST_DIR = "/content/drive/MyDrive/CMPE401/YOLO/data"

In [ ]:
# ── Class mapping ──────────────────────────────────────────────────────────────
VISDRONE_CLASSES = {
    1: 0,   # pedestrian
    2: 1,   # people
    3: 2,   # bicycle
    4: 3,   # car
    5: 4,   # van
    6: 5,   # truck
    7: 6,   # tricycle
    8: 7,   # awning-tricycle
    9: 8,   # bus
    10: 9,  # motor
    # cat 0  = ignored region  → skipped
    # cat 11 = others          → skipped
}

# ── Converter ──────────────────────────────────────────────────────────────────
def convert_split(src_dir, split_name):
    src_dir    = Path(src_dir)
    images_dir = src_dir / "images"
    annots_dir = src_dir / "annotations"

    out_images = Path(DST_DIR) / "images" / split_name
    out_labels = Path(DST_DIR) / "labels" / split_name
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    all_images = sorted(images_dir.glob("*.jpg"))
    total      = len(all_images)

    print(f"\n{'='*60}")
    print(f"  Converting split : {split_name.upper()}")
    print(f"  Source           : {src_dir}")
    print(f"  Images found     : {total}")

    # ── Cache check: skip images already converted ─────────────────────────────
    pending = []
    already_done = 0
    for img_path in all_images:
        out_img   = out_images / img_path.name
        out_label = out_labels / (img_path.stem + ".txt")
        if out_img.exists() and out_label.exists():
            already_done += 1
        else:
            pending.append(img_path)

    if already_done > 0:
        print(f"  Cache hit        : {already_done} / {total} already converted — skipping")
    if not pending:
        print(f"  ✅ Split '{split_name}' fully cached — nothing to do!")
        print(f"{'='*60}")
        return
    print(f"  To process       : {len(pending)} images")
    print(f"{'='*60}")

    # ── Process only pending images ────────────────────────────────────────────
    skipped_imgs  = 0
    total_boxes   = 0
    skipped_boxes = 0

    for idx, img_path in enumerate(pending, 1):
        annot_path = annots_dir / (img_path.stem + ".txt")

        # Progress every 500 images
        if idx % 500 == 0 or idx == len(pending):
            print(f"  [{idx:>5}/{len(pending)}] Processing {img_path.name} ...")

        if not annot_path.exists():
            print(f"  [WARN] No annotation for {img_path.name} — skipping")
            skipped_imgs += 1
            continue

        img          = Image.open(img_path)
        img_w, img_h = img.size

        yolo_lines = []
        with open(annot_path) as f:
            for line in f:
                parts = line.strip().split(",")
                if len(parts) < 6:
                    skipped_boxes += 1
                    continue

                x, y, w, h = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
                score       = int(parts[4])
                cat         = int(parts[5])

                # Skip ignored regions, 'others', and score=0 (ignored annotations)
                if cat not in VISDRONE_CLASSES or score == 0:
                    skipped_boxes += 1
                    continue

                yolo_cat = VISDRONE_CLASSES[cat]
                cx = max(0.0, min(1.0, (x + w / 2) / img_w))
                cy = max(0.0, min(1.0, (y + h / 2) / img_h))
                nw = max(0.0, min(1.0, w / img_w))
                nh = max(0.0, min(1.0, h / img_h))

                if nw > 0 and nh > 0:
                    yolo_lines.append(f"{yolo_cat} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
                    total_boxes += 1
                else:
                    skipped_boxes += 1

        shutil.copy(img_path, out_images / img_path.name)
        with open(out_labels / (img_path.stem + ".txt"), "w") as f:
            f.write("\n".join(yolo_lines))

    print(f"\n  ✅ Done — {split_name.upper()}")
    print(f"     Images converted : {len(pending) - skipped_imgs} / {len(pending)} pending")
    print(f"     Bounding boxes   : {total_boxes} kept  |  {skipped_boxes} skipped (ignored/invalid)")
    print(f"     Output images    : {out_images}")
    print(f"     Output labels    : {out_labels}")


convert_split(TRAIN_DIR_RAW, "train")
convert_split(VAL_DIR_RAW,   "val")
convert_split(TEST_DIR_RAW,  "test")

print(f"\n{'='*60}")
print("  All splits converted successfully!")
print(f"{'='*60}\n")

In [ ]:
yaml_content = f"""\
path: {DST_DIR}
train: images/train
val:   images/val
test:  images/test

nc: 10
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

yaml_path = Path(DST_DIR) / "visdrone.yaml"
yaml_path.parent.mkdir(parents=True, exist_ok=True)
with open(yaml_path, "w") as f:
    f.write(yaml_content)
print(f"  YAML saved → {yaml_path}\n")